# Tugas Kelompok 2: Data Collection Pipeline

## Kelompok sowang

1. Fransisca Ellya Bunaren (2306152286)
2. Micheline Wijaya Limbergh (2306207013)
3. Nisrina Annaisha Sarnadi (2306275960)
4. Rizqya Az Zahra Putri (2306244936)
5. Muhammad Satrio Haryo Kusumo (2006597771)

# Tahap Merge, Validation & Basic Cleaning

**Output** dari tahap ini disimpan di `data/interim/`:
- `merged.csv`: seluruh postingan dari file bulanan, sebelum cleaning.
- `cleaned.csv`: data setelah deduplikasi dan pembuangan empty string.
- `validation_report.json`: statistik validasi dan jumlah data yang dibuang.
- `removed_rows.csv`: data yang dikeluarkan beserta alasannya untuk keperluan audit.

## 0. Persiapan dan konfigurasi

In [2]:
from pathlib import Path
import re, json
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display

CURRENT = Path.cwd().resolve()
ROOT = CURRENT if (CURRENT / "data" / "raw").exists() else CURRENT.parent
RAW_DIR = ROOT / "data" / "raw"
INTERIM_DIR = ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

MIN_WORDS = 3               
DROP_SHORT_POSTS = False   
DEDUP_EXACT_TEXT = True    
DEDUP_CORE_TEXT = True       
EXPECTED_PER_MONTH = 700     
ACCEPTED_LANG = {"id", "in"} 

print("Project root:", ROOT)
print("Raw directory:", RAW_DIR)
print("Drop short posts:", DROP_SHORT_POSTS)

Project root: C:\projects\sowang
Raw directory: C:\projects\sowang\data\raw
Drop short posts: False


## 1. Merge CSV bulanan

- Menjaga ID sebagai teks agar tidak kehilangan presisi.
- Menambahkan `source_month` sebagai asal file.
- Memeriksa **konsistensi skema kolom** sebelum merge.
- Menyimpan hasil sebelum cleaning sebagai `merged.csv`.

In [3]:
files = sorted(p for p in RAW_DIR.glob("posts_????-??.csv") if re.fullmatch(r"posts_\d{4}-\d{2}\.csv", p.name))
if not files:
    raise FileNotFoundError(f"Tidak menemukan posts_YYYY-MM.csv di {RAW_DIR}. Pastikan RAW_DIR sesuai config.py.")

frames, schemas, input_counts = [], {}, {}
for path in files:
    d = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    if len(set(d.columns)) != len(d.columns):
        raise ValueError(f"Nama kolom duplikat pada {path.name}")
    if "source_month" in d.columns:
        raise ValueError(f"Kolom 'source_month' sudah ada di {path.name}; ganti nama metadata agar tidak bentrok")
    schemas[path.name] = list(d.columns)
    input_counts[path.name] = len(d)
    d["source_month"] = path.stem.removeprefix("posts_")
    frames.append(d)

base_columns = schemas[files[0].name]
for filename, columns in schemas.items():
    if set(columns) != set(base_columns):
        raise ValueError(f"Skema kolom berbeda di {filename}: missing={sorted(set(base_columns)-set(columns))}, extra={sorted(set(columns)-set(base_columns))}")

merged = pd.concat([d[base_columns + ["source_month"]] for d in frames], ignore_index=True)
merged = merged.sort_values(["created_at", "id"], kind="stable").reset_index(drop=True)  # BARU
merged.to_csv(INTERIM_DIR / "merged.csv", index=False, encoding="utf-8-sig")
print(f"Merged {len(files)} file, total {len(merged):,} baris → data/interim/merged.csv")
display(pd.DataFrame([{"file": k, "rows": v} for k,v in input_counts.items()]))
display(merged.head(5))

Merged 12 file, total 8,400 baris → data/interim/merged.csv


,file,rows
0,posts_2025-10.csv,700
1,posts_2025-11.csv,700
2,posts_2025-12.csv,700
3,posts_2026-01.csv,700
4,posts_2026-02.csv,700
5,posts_2026-03.csv,700
6,posts_2026-04.csv,700
7,posts_2026-05.csv,700
8,posts_2026-06.csv,700
9,posts_2026-07.csv,700


,month,group,keyword,id,created_at,lang,text,conversation_id,like_count,reply_count,repost_count,collected_at,source_month
0,2025-10,umum,parenting,1973219344600760446,2025-10-01T02:52:07.000Z,in,"Lebih butuh konseling pra- nikah yg mendalam, ...",1973219344600760446,18,2,2,2026-10-25T05:44:22Z,2025-10
1,2025-10,umum,parenting,1973223454599262597,2025-10-01T03:08:26.000Z,in,@toe_giman Miris... Akibat dr pny ortu tp ga b...,1973003932529299748,0,0,0,2026-10-25T05:44:22Z,2025-10
2,2025-10,otoriter,gentle parenting | voc parenting,1973230829380968860,2025-10-01T03:37:45.000Z,in,@sopir_idiot Kalau kayak gini. Ngasitaunya har...,1973225212507459934,11,5,0,2026-10-25T05:43:46Z,2025-10
3,2025-10,umum,parenting,1973234406467969447,2025-10-01T03:51:58.000Z,in,@kitoorah @tanyakanrl mau pemerintah bikin atu...,1969968688389337286,0,0,0,2026-10-25T05:44:22Z,2025-10
4,2025-10,umum,parenting,1973234962532045079,2025-10-01T03:54:10.000Z,in,Nanti deh mau ngumpulin video2 xinlong yg kusu...,1973234962532045079,1,1,0,2026-10-25T05:44:22Z,2025-10


## 2. Validation
Di tahap ini, kami memeriksa kolom wajib, empty string, panjang teks, ID kosong/duplikat, tanggal tidak valid, tanggal tidak sesuai bulan asal, serta teks identik. Postingan bertanda invalid-date **tidak otomatis dibuang** dalam basic cleaning ini.

In [4]:
TEXT_COL = "text"
ID_COL = "id" if "id" in merged.columns else "post_id" if "post_id" in merged.columns else None
DATE_COL = "created_at"
required = [TEXT_COL, DATE_COL]
if ID_COL is None:
    raise ValueError("Kolom ID tidak ditemukan (diharapkan 'id' atau 'post_id'). Cek kolom CSV raw.")
required.append(ID_COL)
missing = [c for c in required if c not in merged.columns]
if missing:
    raise ValueError(f"Kolom wajib tidak tersedia: {missing}")

work = merged.copy()
URL_RE, MENTION_RE = r"https?://\S+", r"@\w+"

text_norm = work[TEXT_COL].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
normalized_exact_text = text_norm.str.casefold()
empty_text = text_norm.eq("")

core_text = (text_norm.str.replace(URL_RE, " ", regex=True)
                      .str.replace(MENTION_RE, " ", regex=True)
                      .str.replace(r"\s+", " ", regex=True).str.strip())
word_count = text_norm.str.split().str.len().fillna(0).astype(int)
core_word_count = core_text.str.split().str.len().fillna(0).astype(int)
short_text = (~empty_text) & core_word_count.lt(MIN_WORDS)
is_reply = text_norm.str.startswith("@")

PUNCT_RE = r"[\u0021-\u002F\u003A-\u0040\u005B-\u0060\u007B-\u007E\u2000-\u206F]" 
core_key = (core_text.str.casefold().str.replace(PUNCT_RE, "", regex=True)
                     .str.replace(r"\s+", " ", regex=True).str.strip())
core_eligible = core_key.ne("") & core_word_count.ge(MIN_WORDS)
duplicate_core = core_eligible & core_key.duplicated(keep="first")

id_clean = work[ID_COL].fillna("").astype(str).str.strip()
empty_id = id_clean.eq("")
duplicate_id = id_clean.ne("") & id_clean.duplicated(keep="first")
duplicate_text = (~empty_text) & normalized_exact_text.duplicated(keep="first")

parsed_date = pd.to_datetime(work[DATE_COL], utc=True, errors="coerce")
invalid_date = parsed_date.isna()
date_month = parsed_date.dt.strftime("%Y-%m")
month_mismatch = (~invalid_date) & date_month.ne(work["source_month"])

bad_lang = ~work["lang"].isin(ACCEPTED_LANG) if "lang" in work.columns else pd.Series(False, index=work.index)
month_col_mismatch = work["month"].ne(work["source_month"]) if "month" in work.columns else pd.Series(False, index=work.index)

validation = {
    "total_merged": len(work), "empty_text": int(empty_text.sum()),
    "short_text_flagged": int(short_text.sum()), "empty_id": int(empty_id.sum()),
    "duplicate_ids": int(duplicate_id.sum()),
    "duplicate_exact_texts": int(duplicate_text.sum()),
    "duplicate_core_texts": int((~duplicate_text & duplicate_core).sum()),
    "invalid_created_at": int(invalid_date.sum()),
    "created_at_month_mismatch": int(month_mismatch.sum()),
    "bad_lang": int(bad_lang.sum()),
    "month_col_mismatch": int(month_col_mismatch.sum()),
    "reply_posts": int(is_reply.sum()),
}
display(pd.DataFrame([{"check": k, "count": v} for k, v in validation.items()]))
print(f"Porsi reply: {is_reply.mean():.1%}")
print("Sampel postingan pendek (belum dibuang), tinjau sebelum DROP_SHORT_POSTS=True:")
n_short = int(short_text.sum())
display(work.loc[short_text, [ID_COL, TEXT_COL, "source_month"]].sample(min(30, n_short), random_state=1) if n_short else "tidak ada")
print("Sampel duplikat 'inti' (URL/mention dibuang):")
display(work.loc[duplicate_core, [ID_COL, TEXT_COL, "source_month"]].head(15))

,check,count
0,total_merged,8400
1,empty_text,0
2,short_text_flagged,24
3,empty_id,0
4,duplicate_ids,0
5,duplicate_exact_texts,4
6,duplicate_core_texts,29
7,invalid_created_at,0
8,created_at_month_mismatch,0
9,bad_lang,0


Porsi reply: 45.0%
Sampel postingan pendek (belum dibuang), tinjau sebelum DROP_SHORT_POSTS=True:


,id,text,source_month
4055,2036492220208181649,@WatchmenID Belajar parenting,2026-03
5858,2064857307570389140,parenting ceunah https://t.co/oGUuD0IxdS,2026-06
1127,1989227973271261266,@pawlonica kajian parenting,2025-11
4483,2044397937082744856,@tanyakanrl buku parenting?,2026-04
5961,2066712913734463934,Palestinian parenting https://t.co/NNQch06H0j,2026-06
5218,2054577638933758011,Parenting sampah https://t.co/8QJ8ezpMXW,2026-05
2914,2018938954616385973,@HumansNoContext Cyberpunk parenting.,2026-02
1150,1989408357141405873,@tattyhassan Terrrpaling parenting,2025-11
1013,1987914711271387157,@rawdawgcomics Asian parenting,2025-11
5929,2066245620898361675,@AKuurap @beyitalks @tanyakanrl Istilahnya co-...,2026-06


Sampel duplikat 'inti' (URL/mention dibuang):


,id,text,source_month
812,1985611406386086035,Selamat Datang Matahariku: Panduan Praktis Jad...,2025-11
854,1986107008358195545,@rrkhamrthaa Bagus banget parenting nya,2025-11
1054,1988337921615897008,Aku wis nyoba kanggo nggawe buku paling apik s...,2025-11
1460,1996235936364159405,@xellbo Bagus juga parenting nya,2025-12
1483,1996565158793998374,💡 Tip parenting: Anak bawah 12 tahun mudah tir...,2025-12
1576,1998330265022951647,@arina_ayunda bagus itu parenting nya,2025-12
1606,1998703943640383495,@ybravv Parenting itu penting ya,2025-12
1652,1999428609409483211,@iraseorita bagus tuh parenting nya,2025-12
1733,2000920957364666688,@myshawti Kok parenting Nikita Willy,2025-12
1859,2002925378281894325,——⚠️PENYEBAB PROMIL GAGAL⁉️——⚠️#au #alternateu...,2025-12


## 3. Basic cleaning
Aturan yang diterapkan ada berikut ini dengan berurutan:
1. Buang **ID kosong** (tidak bisa dilacak) dan teks kosong.
2. Buang duplikat **ID** (simpan kemunculan pertama).
3. Buang duplikat **teks identik** setelah hanya merapikan spasi & huruf besar/kecil (bukan stemming, bukan membuang emoji/negasi).
4. Buang  duplikat **inti**: teks yang sama setelah URL, mention, dan tanda baca diabaikan (emoji dipertahankan, jadi 😭 dan 😡 tidak dianggap sama).
5. Postingan di bawah `MIN_WORDS` **hanya diberi flag** `is_short` (post tanpa kata sama sekali, mis. hanya URL/mention, juga ditandai `is_empty_core`).

`removed_rows.csv` menyimpan baris yang dibuang beserta alasan. Jumlah alasan dihitung berdasarkan urutan keputusan agar tidak menghitung baris yang sama dua kali. Tidak ada URL/emoji/hashtag yang dibersihkan di sini karena hal tersebut akan di-handle pada tahap preprocessing.

In [ ]:
#  Tabel hasil: salin data + kolom penanda (flag); semua baris masih utuh 
cleaned = work.copy()
cleaned["word_count"] = word_count             
cleaned["core_word_count"] = core_word_count   
cleaned["is_short"] = short_text               
cleaned["is_empty_core"] = core_text.eq("")    
cleaned["is_reply"] = is_reply                 
cleaned["is_dup_core"] = duplicate_core        

#  Mencatat alasan baris dibuang 
removal_reason = pd.Series("", index=work.index, dtype="object")   
dup_of = pd.Series("", index=work.index, dtype="object")           

def mark_rows(mask, reason, kept_id=None):
    m = mask & removal_reason.eq("")
    removal_reason.loc[m] = reason
    if kept_id is not None:
        dup_of.loc[m] = kept_id.loc[m]

def dup_among_remaining(key, eligible):
    keep = removal_reason.eq("") & eligible
    mask = pd.Series(False, index=work.index)
    first_id = pd.Series("", index=work.index, dtype="object")
    mask.loc[keep] = key[keep].duplicated(keep="first")
    first_id.loc[keep] = work.loc[keep, ID_COL].groupby(key[keep]).transform("first")
    return mask, first_id

#  Aturan, berurutan 
mark_rows(empty_id, "empty_id")                                  # ID kosong: tidak bisa dilacak
mark_rows(empty_text, "empty_text")                              # empty text: tidak ada konten
mark_rows(duplicate_id, "duplicate_id", id_clean)                # ID sama: simpan kemunculan pertama
if DEDUP_EXACT_TEXT:                                             # teks identik: simpan satu salinan
    m, first = dup_among_remaining(normalized_exact_text, ~empty_text)
    mark_rows(m, "duplicate_text", first)
if DROP_SHORT_POSTS:                                             # postingan pendek hanya ditandai
    mark_rows(short_text, "short_text")
if DEDUP_CORE_TEXT:                                              # duplikat inti (URL/mention/tanda baca diabaikan): simpan satu
    m, first = dup_among_remaining(core_key, core_eligible)
    mark_rows(m, "duplicate_core_text", first)

#  Memisahkan data bersih dan data yang dibuang 
is_removed = removal_reason.ne("")
removed = cleaned.loc[is_removed].copy()
removed["removal_reason"] = removal_reason[is_removed].values
removed["dup_of"] = dup_of[is_removed].values                 
cleaned = cleaned.loc[~is_removed].copy()

# info: seberapa sering teks ini diposting
n_dups = removed.loc[removed["dup_of"].ne(""), "dup_of"].value_counts()
cleaned["n_duplicates_removed"] = cleaned[ID_COL].map(n_dups).fillna(0).astype(int)
# hitung ulang flag terhadap baris yang tersisa (bila DEDUP_CORE_TEXT=True, hasilnya semua False)
cleaned["is_dup_core"] = core_eligible.loc[cleaned.index] & core_key.loc[cleaned.index].duplicated(keep="first")

#  Simpan dan ringkas 
cleaned.to_csv(INTERIM_DIR / "cleaned.csv", index=False, encoding="utf-8-sig")
removed.to_csv(INTERIM_DIR / "removed_rows.csv", index=False, encoding="utf-8-sig")
print(f"Sebelum cleaning: {len(work):,}")
print(f"Dibuang: {len(removed):,}")
print(f"Setelah cleaning: {len(cleaned):,} → data/interim/cleaned.csv")
print("Alasan baris dibuang:")
display(removed["removal_reason"].value_counts().rename_axis("reason").reset_index(name="count"))

Sebelum cleaning: 8,400
Dibuang: 33
Setelah cleaning: 8,367 → data/interim/cleaned.csv
Alasan baris dibuang:


,reason,count
0,duplicate_core_text,29
1,duplicate_text,4


## 4. Simpan validation report
Report akan memuat **jumlah aktual data**, kriteria cleaning, rincian file asal, dan jumlah yang tersisa per bulan.

In [ ]:
import warnings
report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "input_files": input_counts,
    "columns": list(merged.columns),
    "settings": {"min_words": MIN_WORDS, "drop_short_posts": DROP_SHORT_POSTS,
                 "dedup_exact_text": DEDUP_EXACT_TEXT, "dedup_core_text": DEDUP_CORE_TEXT,
                 "accepted_lang": sorted(ACCEPTED_LANG)},
    "validation_before_cleaning": validation,
    "cleaning": {
        "before": len(merged), "removed_total": len(removed),
        "removed_by_reason": {k: int(v) for k, v in removed["removal_reason"].value_counts().items()},
        "after": len(cleaned),
        "remaining_by_month": {k: int(v) for k, v in cleaned["source_month"].value_counts().sort_index().items()},
        "remaining_short_flagged": int(cleaned["is_short"].sum()),
        "remaining_empty_core_flagged": int(cleaned["is_empty_core"].sum()),
        "remaining_core_dup_flagged": int(cleaned["is_dup_core"].sum()),
        "reply_share": round(float(cleaned["is_reply"].mean()), 4),
    },
}
with (INTERIM_DIR / "validation_report.json").open("w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print("Laporan disimpan:", INTERIM_DIR / "validation_report.json")

# asersi integritas
assert len(merged) == len(cleaned) + len(removed), "Jumlah baris tidak konsisten"
if not all(v == EXPECTED_PER_MONTH for v in input_counts.values()):
    warnings.warn(f"Ada file mentah ≠ {EXPECTED_PER_MONTH} baris (mis. budget berhenti lebih awal): {input_counts}")
assert cleaned[ID_COL].is_unique, "ID tidak unik setelah cleaning"
assert validation["bad_lang"] == 0, "Ada post dengan lang di luar ACCEPTED_LANG"
assert validation["created_at_month_mismatch"] == 0, "Ada post di luar bulan file"
if min(report["cleaning"]["remaining_by_month"].values()) < 0.95 * EXPECTED_PER_MONTH:
    warnings.warn("Ada bulan terpangkas >5%; periksa sebelum membandingkan bulan (RQ3)")
print("✓ Semua pemeriksaan integritas lolos")

Laporan disimpan: C:\projects\sowang\data\interim\validation_report.json
✓ Semua pemeriksaan integritas lolos
